# Pacific Climate Risks dataviz - data preprocessing

This notebooks contains Python code used to preprocess and create dataviz dataset from input datasets. <br>.
Input datasets from official datasets in Pacific dataviz challenge, 2026 edition [Pacific Dataviz Challenge 2026](https://pacificdatavizchallenge.org/):
- Population growth
- Crop yield
- Livestock yield
- Sea level anomalies
- Tourist arrivals

<br>
Other datasets:

- Global Mean Sea Level from [NASA Sea Level Change Portal](https://sealevel.nasa.gov/). Link to [Dataset](https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/NASA_SSH_GMSL_INDICATOR/NASA_SSH_GMSL_INDICATOR.txt)
- Population living in low elevation coastal zones (0-10m and 0-20m above sea level) from [Pacific Environment Data Portal](https://pacific-data.sprep.org/dataset/population-living-low-elevation-coastal-zones-0-10m-and-0-20m-above-sea-level). Link to [Dataset](https://pacific-data.sprep.org/resource/population-living-low-elevation-coastal-zones-0-10m-and-0-20m-above-sea-level-data-0)




In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Development Sustainability dataset generation

In [2]:
# load official datasets already including Crop yield, Livestock yield, Sea level anomalies and Tourist arrivals
df = pd.read_csv(r'C:\Temp\pacific_dataviz\SPC,DF_CLIMATE_CHANGE,1.0,complete,2026-08-20 12-32-19.csv')

In [3]:
# show dataset information
print('rows: {}, columns: {}'.format(len(df), len(df.columns)))
print()
print('example first sample:')
display(df.iloc[0])
print()
# available climate Change Indicators
print('Climate Change Indicators frequency:')
display(df[['CLIMATE_CHANGE_INDICATORS', 'Climate Change Indicators']].value_counts())

rows: 16568, columns: 26

example first sample:


STRUCTURE                                                                            DATAFLOW
STRUCTURE_ID                                                       SPC:DF_CLIMATE_CHANGE(1.0)
STRUCTURE_NAME                                                      Climate Change indicators
ACTION                                                                                      I
FREQ                                                                                        A
Frequency                                                                              Annual
CLIMATE_CHANGE_INDICATORS                                          FISH_MNGT_MULT_BILAT_ARGMT
Climate Change Indicators                   Fisheries management measures in place and mul...
GEO_PICT                                                                                   TO
Pacific Island Countries and territories                                                Tonga
TIME_PERIOD                                                 


Climate Change Indicators frequency:


CLIMATE_CHANGE_INDICATORS   Climate Change Indicators                                                                              
ST_ANOM                     Surface Temperature anomalies                                                                              3872
SST_ANOM                    Sea Surface Temperature anomalies                                                                          3696
METEO_MONITOR_NET           Meteorological monitoring network                                                                          1650
FISH_MNGT_MULT_BILAT_ARGMT  Fisheries management measures in place and multilateral and bilateral fisheries management arrangements    1563
RAIN_ANOM                   Precipitation anomalies                                                                                    1034
GHG_EMI_CAPITA              Greenhouse gaz emission per capita                                                                          935
CROP_YIELD                  

In [4]:
# load official dataset including Population growth information
df_pop = pd.read_csv(r'C:\Temp\pacific_dataviz\SPC,DF_NMDI_POP,1.0,complete,2026-08-24 07-06-22.csv')

In [5]:
# dataset includes some disaggregated information, application of filters to use only absoulte population
f = df_pop['SEX'] == '_T'
f &= df_pop['AGE'] == '_T'
f &= df_pop['UNIT_MEASURE'] == 'N'

# Population in dataset includes more GEO_PICT than other dataset for climate change, only the ones in the second one are considered


# reduce number of columns
f_cols = ['GEO_PICT', 'OBS_VALUE', 'TIME_PERIOD']
df_pop = df_pop.loc[f, f_cols]

display(df_pop.head(2))

,GEO_PICT,OBS_VALUE,TIME_PERIOD
3629,MH,44673.0,1990
3630,MH,45603.0,1991


In [6]:
### Load dataset relative to population by elevation
df_elev = pd.read_csv(r'C:\Temp\pacific_dataviz\SPC,DF_POP_LECZ,1.0,complete,2026-08-26 08-59-56.csv')
# filter selected indicator and columns
f = df_elev['UNIT_MEASURE'] == 'PERCENT'
f_cols = ['GEO_PICT', 'TIME_PERIOD', 'ELEVATION', 'OBS_VALUE']
df_elev = df_elev.loc[f, f_cols].copy()
# change from Long format to wide and rename columns
df_elev = pd.pivot_table(df_elev, index=['GEO_PICT', 'TIME_PERIOD'], columns='ELEVATION', aggfunc={'OBS_VALUE':'sum'}).reset_index()
df_elev.columns = [c[0] if c[0] != 'OBS_VALUE' else c[1] for c in df_elev.columns]

# show example
display(df_elev.head(2))

,GEO_PICT,TIME_PERIOD,10M,20M,5M
0,AS,2010,23.0,42.0,11.0
1,AS,2011,23.0,42.0,11.0


In [7]:
# generation of Sustainable Development dataset
# in this dataset it will be integrated information relative to population and resources explotation (crop, livestock, tourism)

# change population dataset from wide format to long
df_sd = pd.melt(df_pop,id_vars=['GEO_PICT', 'TIME_PERIOD'], value_vars='OBS_VALUE',value_name='POP_N')

# integrate LVST_YIELD, CROP_YIELD
cols = ['LVST_YIELD', 'CROP_YIELD', 'TRSM_ARR']
df_f_cols = ['GEO_PICT', 'TIME_PERIOD', 'OBS_VALUE']
for c in cols:
    # filter indicator
    f = df['CLIMATE_CHANGE_INDICATORS'] == c
    df_f = df.loc[f, df_f_cols]
    df_f = df_f.rename(columns={'OBS_VALUE': c})
    # merge information
    df_sd = pd.merge(df_sd, df_f,
                     how='left',
                     left_on = ['GEO_PICT', 'TIME_PERIOD'],
                     right_on = ['GEO_PICT', 'TIME_PERIOD'])

# filter geo_pict by the ones available in the CLIMATE CHANGE dataset for the specific indicators
f = df_sd['GEO_PICT'].isin(pd.unique(df_f['GEO_PICT']))
df_sd = df_sd[f].copy()
# show example
display(df_sd.head(2))

,GEO_PICT,TIME_PERIOD,variable,POP_N,LVST_YIELD,CROP_YIELD,TRSM_ARR
0,MH,1990,OBS_VALUE,44673.0,NaN,NaN,NaN
1,MH,1991,OBS_VALUE,45603.0,NaN,2300.0,NaN


In [8]:
# integrate information from elevations dataset
df_sd = pd.merge(df_sd, df_elev, 
                 how='left',
                 left_on = ['GEO_PICT', 'TIME_PERIOD'],
                 right_on = ['GEO_PICT', 'TIME_PERIOD'])

# integrate absolute population below certain information by combining shares from elevation dataset and absolute population in climate dateset
for elev in ['10M', '20M', '5M']:
    df_sd[elev + '_N'] = df_sd[elev] / 100 * df_sd['POP_N']

display(df_sd.iloc[200:202])

,GEO_PICT,TIME_PERIOD,variable,POP_N,LVST_YIELD,CROP_YIELD,TRSM_ARR,10M,20M,5M,10M_N,20M_N,5M_N
200,AS,2010,OBS_VALUE,55275.0,NaN,NaN,40300.0,23.0,42.0,11.0,12713.25,23215.50,6080.25
201,AS,2011,OBS_VALUE,54944.0,NaN,NaN,41300.0,23.0,42.0,11.0,12637.12,23076.48,6043.84


## decisions
Implementation of different decisions to improve dataset consistency:
- Tourism Arrivals: analysis related with this parameter will be limited upto 2019, there is evident impact from COVID and there is lot of missed data after it.
- Tourism arrivals: filtered for MH and NU, lot of missed data
- 2025 is filtered for all indicators, lot of missed data
- 2024 is filtered for all indicators, change of trend that needs to be confirmed it is consolidated

In [9]:
# application of decisions
f = df_sd['TIME_PERIOD'] < 2024
df_sd = df_sd[f].copy()

f = df_sd['GEO_PICT'].isin(['MH', 'NU'])
f |= df_sd['TIME_PERIOD'] > 2019
df_sd.loc[f, 'TRSM_ARR'] = np.nan
df_sd.loc[f, 'TRSM_ARR_pop_norm'] = np.nan

## Data normalization
Normalization of indicators to show increments respect value in reference period for the respective indicator and region

In [10]:
# definition of normalization period
periods_in_avg = list(range(1993, 2013))

# aggregate all regions in ALL category
tb = pd.pivot_table(df_sd, index='TIME_PERIOD', 
                    aggfunc={c:'sum' for c in  ['LVST_YIELD', 'CROP_YIELD', 'TRSM_ARR', 'POP_N',
                                                                           '10M_N', '20M_N', '5M_N']})
# replace 0 by nan
f = tb == 0
tb[f] = np.nan
for c in  ['LVST_YIELD', 'CROP_YIELD', 'TRSM_ARR']:
    f = tb.index.isin(periods_in_avg)
    tb[c + '_ratio_delta_perc'] = tb[c] / tb.loc[f, c].mean() * 100 - 100

# calculate averages in each country and show range 
msg = '{} => '.format('ALL')
f_per = tb.index.isin(periods_in_avg)
all_avg = dict()
for field in  ['LVST_YIELD', 'CROP_YIELD', 'TRSM_ARR']:
    c_avg = tb.loc[f_per, field].mean()
    c_vals = tb[field] / c_avg * 100 - 100
    all_avg[field] = c_avg
    c_range = (c_vals.min(), c_vals.max())
    msg += '{} : {:0.0f} [{:0.0f} * {:0.0f}] - '.format(field, c_avg, c_range[0], c_range[1])

msg = msg[:-3]
print(msg)
    

for c in pd.unique(df_sd['GEO_PICT']):
    f = df_sd['GEO_PICT'] == c
    f_per = f & df_sd['TIME_PERIOD'].isin(periods_in_avg)
    msg = '{} => '.format(c)
    for field in  ['LVST_YIELD', 'CROP_YIELD', 'TRSM_ARR']:
        c_avg = df_sd.loc[f_per, field].mean()
        c_vals = df_sd.loc[f, field] / c_avg * 100 - 100
        c_range = (c_vals.min(), c_vals.max())
        msg += '{} : {:0.0f} [{:0.0f} * {:0.0f}] - '.format(field, c_avg / all_avg[field] * 100, c_range[0], c_range[1])

    msg = msg[:-3]
    print(msg)

ALL => LVST_YIELD : 17757 [-10 * 29] - CROP_YIELD : 77088 [-9 * 21] - TRSM_ARR : 1653711 [-40 * 104]
MH => LVST_YIELD : nan [nan * nan] - CROP_YIELD : 4 [-82 * 174] - TRSM_ARR : nan [nan * nan]
TO => LVST_YIELD : 7 [-24 * 37] - CROP_YIELD : 9 [-15 * 28] - TRSM_ARR : 3 [-43 * 86]
NC => LVST_YIELD : 2 [-19 * 25] - CROP_YIELD : 10 [-32 * 28] - TRSM_ARR : 12 [-39 * 211]
VU => LVST_YIELD : 8 [-14 * 123] - CROP_YIELD : 5 [-22 * 45] - TRSM_ARR : 9 [-47 * 144]
FJ => LVST_YIELD : 30 [-29 * 51] - CROP_YIELD : 37 [-24 * 41] - TRSM_ARR : 33 [-21 * 91]
AS => LVST_YIELD : nan [nan * nan] - CROP_YIELD : nan [nan * nan] - TRSM_ARR : 3 [-13 * 33]
NU => LVST_YIELD : 16 [-27 * 10] - CROP_YIELD : 2 [-16 * 8] - TRSM_ARR : nan [nan * nan]
PG => LVST_YIELD : 9 [-12 * 30] - CROP_YIELD : 11 [-21 * 27] - TRSM_ARR : 9 [-18 * 44]
WS => LVST_YIELD : 4 [-34 * 34] - CROP_YIELD : 8 [-29 * 6] - TRSM_ARR : 8 [-8 * 44]
KI => LVST_YIELD : 13 [-31 * 51] - CROP_YIELD : 7 [-61 * 102] - TRSM_ARR : 3 [-85 * 157]
MP => LVST_YI

## Output Development Sustainability dataset

In [11]:
# consolidated dataset
out = {k:[] for k in  ['GEO_PICT', 'TIME_PERIOD', 'POP_N', 'LVST_YIELD', 'CROP_YIELD', 'TRSM_ARR',
                      'LVST_YIELD_var', 'CROP_YIELD_var', 'TRSM_ARR_var',
                      '10M_N', '20M_N', '5M_N']}

# fill ALL GEO_PICT
f_per = tb.index.isin(periods_in_avg)


for field in  ['LVST_YIELD', 'CROP_YIELD', 'TRSM_ARR']:
    c_avg = tb.loc[f_per, field].mean()
    c_vals = tb[field].values / c_avg * 100 - 100
    c_range = (c_vals.min(), c_vals.max())
    out[field] += list(tb[field].values)
    out[field + '_var'] += list(c_vals)

out['POP_N'] += list(tb['POP_N'].values)
out['GEO_PICT'] += ['ALL'] * len(c_vals)
out['TIME_PERIOD'] += list(tb.index)
out['10M_N'] += list(tb['10M_N'].values)
out['20M_N'] += list(tb['20M_N'].values)
out['5M_N'] += list(tb['5M_N'].values)

for c in pd.unique(df_sd['GEO_PICT']):
    f = df_sd['GEO_PICT'] == c
    f_per = f & df_sd['TIME_PERIOD'].isin(periods_in_avg)
    for field in  ['LVST_YIELD', 'CROP_YIELD', 'TRSM_ARR']:
        c_avg = df_sd.loc[f_per, field].mean()
        c_vals = df_sd.loc[f, field].values / c_avg * 100 - 100
        c_range = (c_vals.min(), c_vals.max())

        out[field] += list(df_sd.loc[f, field].values)
        out[field + '_var'] += list(c_vals)
        
    out['GEO_PICT'] += [c] * len(c_vals)
    out['TIME_PERIOD'] += list(df_sd.loc[f, 'TIME_PERIOD'].values)
    out['POP_N'] += list(df_sd.loc[f, 'POP_N'].values)
    out['10M_N'] += list(df_sd.loc[f, '10M_N'].values)
    out['20M_N'] += list(df_sd.loc[f, '20M_N'].values)
    out['5M_N'] += list(df_sd.loc[f, '5M_N'].values)
    
out_df = pd.DataFrame(out)
display(out_df.head(2))


,GEO_PICT,TIME_PERIOD,POP_N,LVST_YIELD,CROP_YIELD,TRSM_ARR,LVST_YIELD_var,CROP_YIELD_var,TRSM_ARR_var,10M_N,20M_N,5M_N
0,ALL,1990,5817835.0,18035.8,71349.9,NaN,1.570342,-7.443499,NaN,NaN,NaN,NaN
1,ALL,1991,5969699.0,17207.2,70116.0,NaN,-3.095998,-9.044138,NaN,NaN,NaN,NaN


In [12]:
# save to csv
out_df.to_csv(r'C:\Temp\pacific_dataviz\radar_consolidated_dataset.csv', index=False)

# 2. Pacific Sea Level dataset generation

In [13]:
# filter indicator from climate change dataset
f = df['CLIMATE_CHANGE_INDICATORS'] == 'SEA_LVL'
f_cols = ['GEO_PICT', 'TIME_PERIOD', 'OBS_VALUE']
df_sealvl = df.loc[f, f_cols].copy()

# calculate average from all regions and P10, P90 percentile values
tb = pd.pivot_table(df_sealvl, index=['TIME_PERIOD'], aggfunc={'OBS_VALUE':['mean', lambda x: np.quantile(x, 0.1),
                                                                           lambda x: np.quantile(x, 0.9)]})
tb.columns = ['P10', 'P90', 'MEAN']
tb = tb.reset_index()

display(tb.head(2))
# save to csc
tb.to_csv(r'C:\Temp\pacific_dataviz\local_sealevel_consolidated_dataset.csv', index=False)

,TIME_PERIOD,P10,P90,MEAN
0,1993,-0.1,0.0,-0.019048
1,1994,0.0,0.0,-0.004762


# 3. Global Sea Level dataset generation

In [14]:
# load dataset from file
df_glob_sealvl = pd.read_csv(r'C:\Temp\pacific_dataviz\NASA_SSH_GMSL_INDICATOR.txt', skiprows=42, sep='\s+', header=0,
                             names=['TIME_PERIOD', 'obs', 'obs_smooth'])

# extract smoothed values for first record in the year
_glob_sealvl = df_glob_sealvl.loc[df_glob_sealvl['TIME_PERIOD'].astype(int).drop_duplicates().index].copy()
df_glob_sealvl['TIME_PERIOD'] = df_glob_sealvl['TIME_PERIOD'].astype(int)

# normalize values to absolute variation from average value in reference period
f_per = df_glob_sealvl['TIME_PERIOD'].isin(periods_in_avg)
per_avg = df_glob_sealvl.loc[f_per, 'obs_smooth'].mean()
df_glob_sealvl['SEA_LVL'] = df_glob_sealvl['obs_smooth'] - per_avg
df_glob_sealvl = df_glob_sealvl[['TIME_PERIOD', 'SEA_LVL']]

# save dataset to .csv
df_glob_sealvl.to_csv(r'C:\Temp\pacific_dataviz\global_sealevel_consolidated_dataset.csv', index=False)

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Z003ZBWR\AppData\Local\Temp\ipykernel_15604\1740482730.py:2: SyntaxWarning: invalid escape sequence '\s'
  df_glob_sealvl = pd.read_csv(r'C:\Temp\pacific_dataviz\NASA_SSH_GMSL_INDICATOR.txt', skiprows=42, sep='\s+', header=0,
